# Item Selection and Stopping Rules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/04_item_selection_and_stopping.ipynb)

After you understand the basic architecture, the next question is usually:
which CAT strategies should I start with? This notebook gives a compact
comparison of common initializer, selector, estimator, and stopper choices.

In [ ]:
import matplotlib

matplotlib.use("Agg")

from operator import itemgetter

import numpy as np

from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer, RandomInitializer
from catsim.item_bank import ItemBank
from catsim.selection import AStratSelector, LinearSelector, MaxInfoSelector, RandomSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import ConfidenceIntervalStopper, MinErrorStopper, TestLengthStopper

In [ ]:
item_bank = ItemBank.generate_item_bank(160, seed=23)
examinees = np.linspace(-2.5, 2.5, 50)

## Strategy taxonomy

In a CAT loop:

- the initializer chooses the first ability estimate
- the selector chooses the next item
- the estimator updates ability after each response
- the stopper decides when the session ends

Different combinations change precision, test length, and exposure behavior.

## Compare a few selectors under a fixed-length CAT

In [ ]:
selector_rows = []
selector_configs = {
  "max_info": MaxInfoSelector(),
  "random": RandomSelector(),
  "linear": LinearSelector(list(range(15))),
  "a_strat": AStratSelector(test_size=15),
}

for name, selector in selector_configs.items():
  result = SimulationRunner(
    item_bank=item_bank,
    initializer=FixedPointInitializer(0.0),
    selector=selector,
    estimator=NumericalSearchEstimator(),
    stopper=TestLengthStopper(max_items=15),
    seed=77,
  ).run(examinees)
  selector_rows.append({
    "selector": name,
    "rmse": result.rmse,
    "bias": result.bias,
    "overlap_rate": result.overlap_rate,
  })

for row in sorted(selector_rows, key=itemgetter("rmse")):
  print({key: round(float(value), 3) if key != "selector" else value for key, value in row.items()})

## Compare stopping rules with the same selector

In [ ]:
stopper_rows = []
stopper_configs = {
  "fixed_length": TestLengthStopper(max_items=15),
  "min_error": MinErrorStopper(0.35, min_items=6, max_items=20),
  "confidence_interval": ConfidenceIntervalStopper(
    [-3.0, -1.0, 0.0, 1.0, 3.0],
    confidence=0.95,
    min_items=6,
    max_items=20,
  ),
}

for name, stopper in stopper_configs.items():
  result = SimulationRunner(
    item_bank=item_bank,
    initializer=RandomInitializer(),
    selector=MaxInfoSelector(),
    estimator=NumericalSearchEstimator(),
    stopper=stopper,
    seed=88,
  ).run(examinees)
  stopper_rows.append({
    "stopper": name,
    "rmse": result.rmse,
    "bias": result.bias,
    "mean_length": np.mean([session.administered_count for session in result.sessions]),
  })

for row in sorted(stopper_rows, key=itemgetter("rmse")):
  print({key: round(float(value), 3) if key != "stopper" else value for key, value in row.items()})

## Practical guidance

- `MaxInfoSelector` is a strong baseline when you want a classic adaptive CAT.
- `RandomSelector` and `LinearSelector` are useful baselines, not usually final production choices.
- Stratified selectors are useful when you want more structured item-bank usage.
- Fixed-length rules are simpler to compare.
- Precision-based rules such as `MinErrorStopper` are often more realistic for adaptive delivery.

This notebook is meant to narrow the search space, not to replace a proper
methodological evaluation on your own item bank and population.